In [1]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv("../.env")                      # notebook runs in backend/notebooks/
client = genai.Client(api_key=os.environ["CONTEXTFORGE_GEMINI_API_KEY"])

In [2]:
resp = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Explain Python's GIL in two sentences.",
)
print(resp.text)
print(resp.usage_metadata)

The Global Interpreter Lock (GIL) is a mutex in CPython that ensures only one thread executes Python bytecode at a time. While it simplifies memory management and C-extension integration, it prevents multi-threaded Python programs from utilizing multiple CPU cores simultaneously.
cache_tokens_details=None cached_content_token_count=None candidates_token_count=50 candidates_tokens_details=None prompt_token_count=10 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=10
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=60 traffic_type=None


In [3]:
from datasets import load_dataset

ds = load_dataset("google/frames-benchmark")
df = ds["test"].to_pandas()
df.shape

(824, 16)

In [8]:
q = df.iloc[0]
resp = client.models.generate_content(model="gemini-3.8-flash", contents=q["Prompt"])
print("Q:", q["Prompt"][:200])
print("Gemini:", resp.text[:500])
print("Gold:", q["Answer"])

Q: If my future wife has the same first name as the 15th first lady of the United States' mother and her surname is the same as the second assassinated president's mother's maiden name, what is my future
Gemini: To find your future wife's name, we can break down the two parts:

1. **First Name:** 
   * The 15th President was bachelor James Buchanan, so his niece, **Harriet Lane**, served as the 15th First Lady of the United States. 
   * Harriet Lane's mother was **Jane** Ann Buchanan (James Buchanan's sister). 
   * Therefore, her first name is **Jane**.

2. **Surname:** 
   * The four assassinated U.S. presidents in chronological order are Abraham Lincoln (1st), James A. Garfield (2nd), William McKinl
Gold: Jane Ballou


In [7]:
from google.genai import errors

def ask(prompt, models=("gemini-3.8-flash", "gemini-3.6-flash", "gemini-3.5-flash-lite")):
    for m in models:
        try:
            r = client.models.generate_content(model=m, contents=prompt)
            print("answered by", m)
            return r
        except errors.ServerError as e:      # 5xx = Google busy → try next model
            print(m, "busy:", e.code)
    raise RuntimeError("all models busy")

resp = ask(q["Prompt"])
print(resp.text[:500]); print("Gold:", q["Answer"])

KeyboardInterrupt: 